In [1]:
import os
os.chdir('../')
%pwd

'/home/minh_khai/salinity/salinity-ml'

In [2]:
from dataclasses import dataclass
from pathlib import Path

from core.models.configs import (
    ModelConfig, XGBoostConfig, LSTMConfig, iTransformerConfig
)

@dataclass(frozen=True)
class EvaluationModelConfig:
    model_name:  str    
    window_size: int
    horizon:     int
    model_dir:   Path
    onnx_dir:    Path

ModelSpecificConfig = XGBoostConfig | LSTMConfig | iTransformerConfig

@dataclass(frozen=True)
class EvaluationConfig:
    root_dir:       Path
    data_dir:       Path
    encoders_dir:   Path
    
    model:        EvaluationModelConfig
    model_config: ModelSpecificConfig

In [3]:
from core.constants import *
from core import read_yaml, create_directories


class ConfigurationManager:
    def __init__(self, 
                 config_filepath = CONFIG_FILE_PATH, 
                 params_filepath = PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])
    
    _CONFIG_BUILDERS = {
        'xgboost': '_get_xgboost_config',
        'lstm':    '_get_lstm_config',
        'iTransformer': '_get_itransformer_config',
    }
    
    def _get_model_config(self, model_name: str) -> ModelSpecificConfig:
        builder_name = self._CONFIG_BUILDERS.get(model_name)
        if not builder_name:
            raise ValueError(
                f"Unsupported model: '{model_name}'. "
                f"Choose from: {list(self._CONFIG_BUILDERS)}"
            )
        return getattr(self, builder_name)()
    
    def _get_xgboost_config(self) -> XGBoostConfig:
        p = self.params.xgboost
        return XGBoostConfig(
            n_estimators   = p.n_estimators,
            max_depth      = p.max_depth,
            learning_rate  = p.learning_rate,
            subsample      = p.subsample
        )
    
    def _get_lstm_config(self) -> LSTMConfig:
        p = self.params.lstm
        return LSTMConfig(
            hidden_size    = p.hidden_size,
            num_layers     = p.num_layers,
            dropout        = p.dropout,
            
            epochs         = p.epochs,
            batch_size     = p.batch_size,
            learning_rate  = p.learning_rate,
            patience       = p.patience,
        )
    
    def _get_itransformer_config(self) -> iTransformerConfig:
        p = self.params.iTransformer
        return iTransformerConfig(
            hidden_size    = p.hidden_size,
            epochs         = p.epochs,
            batch_size     = p.batch_size,
            learning_rate  = p.learning_rate,
            patience       = p.patience,
        )
        
    def get_evaluation_config(self) -> EvaluationConfig:
        eval_config  = self.config.evaluation_config
        train_config = self.config.training_config
        train_params = self.params.training_params
        create_directories([eval_config.root_dir])
        
        return EvaluationConfig(
            root_dir = Path(eval_config.root_dir),
            encoders_dir = Path(train_config.encoders_dir),
            data_dir = Path(self.config.transformation_config.root_dir) / "infer",
            
            model    = EvaluationModelConfig(
                model_name  = train_params.model_name,
                window_size = train_params.sliding_window_size,
                horizon     = train_params.sliding_horizon,
                model_dir   = Path(train_config.root_dir) / train_params.model_name,
                onnx_dir    = Path(train_config.root_dir) / train_params.model_name,
            ),
            
            model_config = self._get_model_config(model_name=train_params.model_name),
        )

In [8]:
import os, sys, json, time, glob
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.metrics import (
    mean_absolute_error, root_mean_squared_error, r2_score
)

from core.logging import logger
from core.exception import CustomException
from core.data.process import preprocess
from core.data.sliding_window import sliding


from core.models.xgboost_strategy import XGBoostStrategy
from core.models.lstm_strategy import LSTMStrategy
from core.models.iTransformer_strategy import iTransformerStrategy


class Evaluation:
    def __init__(self, config: EvaluationConfig):
        self.config = config
        
    # ─────────────────────────────────────────────
    # Data
    # ─────────────────────────────────────────────
    def _get_infer_data(self):
        train_files = glob.glob(os.path.join(Path(self.config.data_dir).parent / "train", "*.csv"))
        valid_files = glob.glob(os.path.join(Path(self.config.data_dir).parent / "valid", "*.csv"))
        infer_files = glob.glob(os.path.join(self.config.data_dir, "*.csv"))

        full_df = pd.concat([
            pd.concat([pd.read_csv(f) for f in train_files], ignore_index=True),
            pd.concat([pd.read_csv(f) for f in valid_files], ignore_index=True),
            pd.concat([pd.read_csv(f) for f in infer_files], ignore_index=True),
        ], ignore_index=True)
        full_df = full_df.sort_values("timestamp").reset_index(drop=True)

        clean_df = preprocess(
            full_df, out_dir=str(self.config.encoders_dir), fit_encoders=False
        )

        X, y, _, _ = sliding(
            clean_df,
            train_cutoff = pd.to_datetime(full_df["timestamp"]).max(),
            model_type   = self.config.model.model_name,
            window_size  = self.config.model.window_size,
            horizon      = self.config.model.horizon,
        )
        return X, y
    
    # ─────────────────────────────────────────────
    # Load
    # ─────────────────────────────────────────────
    def _load_metadata(self) -> dict:
        path = self.config.model.model_dir / "metadata.json"
        if not path.exists():
            raise FileNotFoundError(f"metadata.json not found at {path}")
        with open(path) as f:
            return json.load(f)
        
    def _load_strategy(self):
        name = self.config.model.model_name
        metadata = self._load_metadata()
        
        model_config = ModelConfig(
            model_name  = name,
            window_size = self.config.model.window_size,
            horizon     = self.config.model.horizon,
        )
        
        if name == "xgboost":
            strategy = XGBoostStrategy(self.config.model_config)
        elif name == "lstm":
            strategy = LSTMStrategy(self.config.model_config, model_config, input_size=metadata["input_size"])
        elif name == "iTransformer":
            strategy = iTransformerStrategy(self.config.model_config, model_config)
        else:
            raise ValueError(f"Unsupported model: {name}")
        
        strategy.load(self.config.model.model_dir)
        return strategy
        
    def _load_onnx(self, use_cuda: bool = False):
        import onnxruntime as ort
        path = self.config.model.onnx_dir / "model.onnx"
        if not path.exists():
            logger.warning(f"ONNX not found: {path}")
            return None
        
        if use_cuda:
            if "CUDAExecutionProvider" not in ort.get_available_providers():
                logger.warning("CUDA not available — skipping CUDA ONNX")
                return None
            return ort.InferenceSession(str(path), providers=["CUDAExecutionProvider"])

        return ort.InferenceSession(str(path), providers=["CPUExecutionProvider"])

    # ─────────────────────────────────────────────
    # Metrics
    # ─────────────────────────────────────────────
    def _compute_metrics(self, preds: np.ndarray, y: np.ndarray, elapsed_ms: float) -> dict:
        return {
            "mae":  round(float(mean_absolute_error(y, preds)), 4),
            "rmse": round(float(root_mean_squared_error(y, preds)), 4),
            "r2":   round(float(r2_score(y, preds)), 4),
            "ms":   round(elapsed_ms, 2),
        }
        
    # ─────────────────────────────────────────────
    # Eval Helpers
    # ─────────────────────────────────────────────   
    def _eval_native(self, X, y) -> dict:
        strategy = self._load_strategy()
        
        start    = time.perf_counter()
        preds    = strategy.predict(X)
        elapsed  = (time.perf_counter() - start) * 1000
        
        return self._compute_metrics(preds, y, elapsed)
    
    def _eval_onnx(self, X, y, use_cuda: bool = False) -> dict | None:
        session  = self._load_onnx(use_cuda)
        if session is None: 
            return None
        
        start      = time.perf_counter()
        ort_inputs = {session.get_inputs()[0].name: X.astype(np.float32)}
        preds      = session.run(None, ort_inputs)[0]
        elapsed    = (time.perf_counter() - start) * 1000
        
        return self._compute_metrics(preds, y, elapsed)
    
    def _print_table(self, results: dict):
        print(f"\n{'Model':<20} {'MAE':>8} {'RMSE':>8} {'R2':>8} {'ms':>10}")
        print("-" * 56)
        for name, m in results.items():
            print(f"{name:<20} {m['mae']:>8.4f} {m['rmse']:>8.4f} {m['r2']:>8.4f} {m.get('ms', 0):>10.2f}")

    def _save_metrics(self, results: dict):
        path = self.config.root_dir / "metrics.json"
        with open(path, "w") as f:
            json.dump(results, f, indent=4)
        logger.info(f"Metrics saved -> {path}")
    
    
    # ─────────────────────────────────────────────
    # Evaluation
    # ─────────────────────────────────────────────
    def evaluate(self) -> dict:
        try:
            X, y    = self._get_infer_data()
            results = {}
            name    = self.config.model.model_name
            
            logger.info("Evaluating native model...")
            results["native"] = self._eval_native(X, y)
            
            if name != "xgboost":
                logger.info("Evaluating ONNX CPU...")
                onnx_cpu = self._eval_onnx(X, y, use_cuda=False)
                if onnx_cpu:
                    results["onnx_cpu"] = onnx_cpu

                logger.info("Evaluating ONNX CUDA...")
                onnx_cuda = self._eval_onnx(X, y, use_cuda=True)
                if onnx_cuda:
                    results["onnx_cuda"] = onnx_cuda
            
            self._print_table(results)
            self._save_metrics(results)
            return results
            
        except Exception as e:
            raise CustomException(e, sys)

In [9]:
import sys
import mlflow
from core.exception import CustomException

try:
    cfg_manager = ConfigurationManager()
    config      = cfg_manager.get_evaluation_config()
    evaluation  = Evaluation(config=config)
    
    with mlflow.start_run(run_name=config.model.model_name):
        metrics = evaluation.evaluate()
        print(metrics)

except Exception as e:
    raise CustomException(e, sys)

[2026-05-25 04:37:47,068: INFO: __init__: yaml file: config/config.yaml loaded successfully]
[2026-05-25 04:37:47,072: INFO: __init__: yaml file: params.yaml loaded successfully]
[2026-05-25 04:37:47,074: INFO: __init__: created directory at: artifacts]
[2026-05-25 04:37:47,076: INFO: __init__: created directory at: artifacts/evaluation]
[2026-05-25 04:37:47,175: INFO: 1085638395: Evaluating native model...]
[2026-05-25 04:37:53,161: INFO: 1085638395: Evaluating ONNX CPU...]
[2026-05-25 04:37:53,205: INFO: 1085638395: Evaluating ONNX CUDA...]


2026-05-25 04:37:53.182672615 [W:onnxruntime:Default, device_discovery.cc:283 GetGpuDevices] Failed to detect devices under "/sys/class/drm/card0": device_discovery.cc:93 ReadFileContents Failed to open file: "/sys/class/drm/card0/device/vendor"
2026-05-25 04:37:53.222568430 [W:onnxruntime:, session_state.cc:1367 VerifyEachNodeIsAssignedToAnEp] Some nodes were not assigned to the preferred execution providers which may or may not have an negative impact on performance. e.g. ORT explicitly assigns shape related ops to CPU to improve perf.
2026-05-25 04:37:53.222603695 [W:onnxruntime:, session_state.cc:1369 VerifyEachNodeIsAssignedToAnEp] Rerunning with verbose output on a non-minimal build will show node assignments.



Model                     MAE     RMSE       R2         ms
--------------------------------------------------------
native                 2.4199   3.3394   0.8760     715.06
onnx_cpu               2.4199   3.3395   0.8760       2.09
onnx_cuda              2.4199   3.3395   0.8760     170.39
[2026-05-25 04:37:53,863: INFO: 1085638395: Metrics saved -> artifacts/evaluation/metrics.json]
{'native': {'mae': 2.4199, 'rmse': 3.3394, 'r2': 0.876, 'ms': 715.06}, 'onnx_cpu': {'mae': 2.4199, 'rmse': 3.3395, 'r2': 0.876, 'ms': 2.09}, 'onnx_cuda': {'mae': 2.4199, 'rmse': 3.3395, 'r2': 0.876, 'ms': 170.39}}
